# 00 Data Catalog

Inventory the raw Toast POS and weather files, inspect shapes and columns, and confirm basic date coverage before deeper analysis.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'notebooks' / 'lib'))

import pantry_eda as eda

pd = eda.pd
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)

## Raw File Inventory

In [ ]:
files = eda.find_files()
files = files[~files['path'].str.contains('xtraCHEF', case=False, na=False)].copy()
files

## Load Core Exports

In [ ]:
sales = eda.read_pos_item_selections()
modifiers = eda.read_pos_modifiers()
weather = eda.read_weather_daily(year=2026)
product_mix = eda.read_product_mix_export()

summary = pd.DataFrame([
    {'dataset': 'POS item selections', 'rows': len(sales), 'columns': sales.shape[1]},
    {'dataset': 'POS modifiers', 'rows': len(modifiers), 'columns': modifiers.shape[1]},
    {'dataset': 'Chicago daily weather, 2026 to date', 'rows': len(weather), 'columns': weather.shape[1]},
    *[{'dataset': f'Product Mix: {name}', 'rows': len(df), 'columns': df.shape[1]} for name, df in product_mix.items()],
])
summary

## Date Coverage

In [ ]:
pd.DataFrame([
    {
        'dataset': 'POS item selections',
        'min_date': sales['business_date'].min(),
        'max_date': sales['business_date'].max(),
        'distinct_dates': sales['business_date'].nunique(),
    },
    {
        'dataset': 'POS modifiers',
        'min_date': modifiers['business_date'].min(),
        'max_date': modifiers['business_date'].max(),
        'distinct_dates': modifiers['business_date'].nunique(),
    },
    {
        'dataset': 'Chicago daily weather',
        'min_date': weather['business_date'].min(),
        'max_date': weather['business_date'].max(),
        'distinct_dates': weather['business_date'].nunique(),
    },
])

## Column Reference

In [ ]:
for name, df in {
    'sales': sales,
    'modifiers': modifiers,
    'weather': weather,
}.items():
    print(f'\n{name}')
    print(list(df.columns))

## Product Mix Files

The Product Mix export contains several report tables. The current Jan-Apr export is useful for longer-range menu/category mix, but it does not include daily rows.

In [ ]:
for name, frame in product_mix.items():
    print(name, frame.shape)
    display(frame.head(10))